In [6]:
#install libraries
!pip install -q transformers accelerate datasets peft trl bitsandbytes sentencepiece pandas==2.2.2
#import libraries
import pandas as pd
import torch #the engine that runs the neural network math
import time #for tracking processing time
import gc #garbage Collector to clean up GPU memory

#connect to my Google Drive to save the trained model permanently
from google.colab import drive
drive.mount('/content/drive')

#import specific tools for loading datasets and managing AI models
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
#import PEFT (Parameter-Efficient Fine-Tuning) for the QLoRA method
from peft import LoraConfig, prepare_model_for_kbit_training, PeftModel
#import the trainer that simplifies the Fine-tuning process
from trl import SFTTrainer, SFTConfig

#setup
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

TRAIN_FILE = "/content/drive/MyDrive/data/train_short.jsonl"
TEST_FILE = "/content/drive/MyDrive/data/dataset_clean.csv"

#where to save the learned knowledge (adapters) on Google Drive
OUTPUT_DIR = "/content/drive/MyDrive/data/qlora_tax_model"
ADAPTER_DIR = f"{OUTPUT_DIR}/adapter"

SYSTEM_PROMPT = """Du bist ein Experte für österreichisches Steuerrecht mit fundierten Kenntnissen in:
- EStG 1988 (Einkommensteuergesetz)
- UStG 1994 (Umsatzsteuergesetz)
- KStG (Körperschaftsteuergesetz)
- GrEStG 1987 (Grunderwerbsteuergesetz)

Beantworte Fragen auf Deutsch, präzise, fachlich korrekt und im Stil einer steuerrechtlichen Musterlösung.
Regeln:
- Antworte direkt ohne Einleitung.
- Verwende klare juristische Formulierungen.
- Zitiere immer die relevanten gesetzlichen Bestimmungen im Format: § [number] Abs. [x] [Gesetz].
- Führe Ausnahmen an, wenn diese bestehen.
- Vermeide Umgangssprache, Aufzählungen und Markdown-Formatierung.
- Antworte in 1–4 prägnanten Sätzen.
"""
#print status and check if the GPU is connected and recognized
print("Setup done")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup done
CUDA available: True
GPU: Tesla T4


In [8]:
#load training dataset
train_data = load_dataset("json", data_files="/content/drive/MyDrive/data/train_short.jsonl", split="train")
print(f"Total training rows loaded: {len(train_data)}")
print(train_data[0])

Total training rows loaded: 92
{'messages': [{'role': 'system', 'content': 'Du bist ein Experte für österreichisches Steuerrecht mit fundierten Kenntnissen in:\n- EStG 1988 (Einkommensteuergesetz)\n- UStG 1994 (Umsatzsteuergesetz)\n- KStG (Körperschaftsteuergesetz)\n- GrEStG 1987 (Grunderwerbsteuergesetz)\n\nBeantworte Fragen auf Deutsch, präzise, fachlich korrekt und im Stil einer steuerrechtlichen Musterlösung.\nRegeln:\n- Antworte direkt ohne Einleitung.\n- Verwende klare juristische Formulierungen.\n- Zitiere immer die relevanten gesetzlichen Bestimmungen im Format: § [number] Abs. [x] [Gesetz].\n- Führe Ausnahmen an, wenn diese bestehen.\n- Vermeide Umgangssprache, Aufzählungen und Markdown-Formatierung.\n- Antworte in 1–4 prägnanten Sätzen.'}, {'role': 'user', 'content': 'Wer ist in Österreich einkommensteuerpflichtig?'}, {'role': 'assistant', 'content': 'Nur natürliche Personen sind einkommensteuerpflichtig (§ 1 EStG); juristische Personen unterliegen stattdessen der Körperschaf

In [9]:
#load tokenizer and model
#determine the best mathematical precision: use bfloat16 if the GPU supports it, otherwise float16
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

#configure 4-bit quantization (QLoRA) to compress the model significantly
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, #enable 4-bit loading to reduce memory usage by ~75%
    bnb_4bit_quant_type="nf4", #use 'Normal Float 4', a specialized data type for better accuracy
    bnb_4bit_use_double_quant=True, #compress the quantization constants themselves for extra space saving
    bnb_4bit_compute_dtype=compute_dtype #use the precision we defined above for calculations
)

#load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)

#if the model doesn't have a specific padding token, use the end-of-sentence token instead
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# align text to the right when adding padding (important for technical model reasons)
tokenizer.padding_side = "right"

#load the actual AI model with the compression settings (bnb_config)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config, #apply the 4-bit compression defined earlier
    device_map="auto", #automatically place the model on the available GPU
    trust_remote_code=True #allow the model to run custom code from the authors
)

#disable cache during training because it's not compatible with weight updates
model.config.use_cache = False
#prepare the compressed model specifically for k-bit training (Gradient Checkpointing, etc.)
model = prepare_model_for_kbit_training(model)

print("Model loaded")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded


In [10]:
#lora config
peft_config = LoraConfig(
    r=8, #rank or size of the update matrices. 8 is a standard balance between speed and quality
    lora_alpha=16, #a scaling factor for the learning process. Usually set to 2x the rank (r)
    lora_dropout=0.05, #helps prevent overfitting by randomly ignoring 5% of neurons during training
    bias="none", #do not train the bias parameters to keep the update as small and fast as possible
    task_type="CAUSAL_LM", #specify the type of model: Causal Language Modeling (predicting the next word)
    target_modules=["q_proj", "v_proj"] #specify exactly which parts of the Transformer 'Brain' to update (Attention layers), q_proj (Query) and v_proj (Value)
)

print("LoRA config ready")

LoRA config ready


In [11]:
#training config
training_args = SFTConfig(
    output_dir=OUTPUT_DIR, #folder on Google Drive where the trained model will be saved
    num_train_epochs=1, #the model will read your entire tax dataset exactly 1 time
    per_device_train_batch_size=1, #process 1 question at a time to keep GPU memory usage very low
    gradient_accumulation_steps=16, #wait for 16 questions before updating weights; creates a stable learning process
    learning_rate=2e-4, #the speed of learning: how much the weights change per step
    logging_steps=20, #every 20 steps, print the training status (Loss, progress)
    save_strategy="steps", #save copies of the model periodically during training
    save_steps=100, #create a save point every 100 steps so you don't lose progress
    save_total_limit=2, #only keep the 2 most recent save points to save Google Drive space
    optim="paged_adamw_8bit", #a memory-efficient "optimizer" that updates weights using 8-bit precision
    bf16=torch.cuda.is_bf16_supported(), #use faster bfloat16 math if your GPU supports it
    fp16=not torch.cuda.is_bf16_supported(), #use float16 math as a fallback for older GPUs
    gradient_checkpointing=True, #trade speed for memory: recalculates parts of the math to avoid crashes
    report_to="none", #don't send training data to external websites
    max_length=256, #cut off any question/answer that is longer than 256 tokens
    packing=False, #not combine multiple short questions into one long line
    eos_token="<|im_end|>" #the specific signal that tells the model the answer is finished
)

trainer = SFTTrainer(
    model=model, #the base AI "brain" we loaded earlier
    args=training_args, #the training parameters we just defined
    train_dataset=train_data, # tax dataset (train_short.jsonl)
    processing_class=tokenizer, #the tool that converts text to numbers
    peft_config=peft_config #the LoRA settings (the small layer we are actually training)
)

print("Trainer ready")

Tokenizing train dataset:   0%|          | 0/92 [00:00<?, ? examples/s]

Trainer ready


In [12]:
#start training
from transformers.trainer_utils import get_last_checkpoint #import a utility to check if the training was interrupted and can be continued

print("Starting fine-tuning...")

#check Google Drive folder to see if any previous checkpoints exist
last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

#if a save point is found, resume training from where it left off to save time
if last_checkpoint is not None:
    print("Resuming from checkpoint:", last_checkpoint)
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("No checkpoint found, starting from scratch") #if no previous progress is found, start the learning process from the beginning
    trainer.train()

print("Fine-tuning finished")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting fine-tuning...
No checkpoint found, starting from scratch


Step,Training Loss


Fine-tuning finished


In [13]:
#save adapter
trainer.model.save_pretrained(ADAPTER_DIR) #save the Adapter weights (the small specialized layer that contains the tax law knowledge)
tokenizer.save_pretrained(ADAPTER_DIR) #save the Tokenizer settings to ensure the model always understands words the same way

print("Adapter saved")
print(ADAPTER_DIR)

Adapter saved
/content/drive/MyDrive/data/qlora_tax_model/adapter


In [14]:
#clear memory
#remove the trainer object from memory to free up space after finishing fine-tuning
del trainer
gc.collect()
torch.cuda.empty_cache() #clear the cache in the GPU so that new models or tasks can use the memory

print("Memory cleared")

Memory cleared


In [15]:
#load fine-tuned model for inference
#load the Tokenizer from the Adapter folder to ensure text is processed exactly as it was during training
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)

#ensure there is a padding token; if missing, use the 'end-of-string' token to fill gaps
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

#load the "empty" Base Model (the original Qwen) using the same 4-bit compression (bnb_config)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
#use PEFT to "snap" your trained Adapter onto the Base Model, creating the specialized Tax Expert
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval() #set the model to 'evaluation' mode, which turns off training features for faster and stable answers

print("Fine-tuned model loaded for inference")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Fine-tuned model loaded for inference


In [16]:
#load test dataset
df = pd.read_csv(TEST_FILE, encoding="utf-8-sig")
print(f"Total questions loaded: {len(df)}")
print(df.head(3))

Total questions loaded: 643
             id                                             prompt
0  CORP-TAX-001  Was ist die steuerliche Bemessungsgrundlage fü...
1  CORP-TAX-002  Welche steuerlichen Konsequenzen hat es, wenn ...
2  CORP-TAX-003  Welche Körperschaften sind verpflichtet, sämtl...


In [21]:
#function to get answer and error handling
def get_answer(question):
    try: #create the standard message format with our "Lawyer Persona" and the user's question
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ]
        # use the 'Chat Template' to wrap the messages in the specific format the Qwen model was trained on
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False, #we want the raw text first before converting to numbers
            add_generation_prompt=True
        )

        inputs = tokenizer(text, return_tensors="pt") #convert the formatted text into Tensors and move them to the GPU
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        # disable gradient calculations to save memory and speed up the generation
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=100, #limit the answer to 100 words to keep it concise
                do_sample=False, #greedy decoding
                pad_token_id=tokenizer.pad_token_id #use the correct signal for empty space
            )
        #decode the numbers back into human-readable text, skipping the input part of the sequence
        answer = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True #remove technical tags like <|im_end|> from the final string
        ).strip()
        #if the model produces an empty string, label it as an ERROR for dataset
        if answer == "":
            return "ERROR"

        return answer

    except Exception as e: #if any technical error occurs, print it and wait 10 seconds before continuing
        print(f"Error: {e}")
        print("  Waiting 10 seconds...")
        time.sleep(10)
        return "ERROR"

In [22]:
#main loop

answers = []
ids = []

#loop through every row in the 'dataset_clean.csv' file
for i, row in df.iterrows():
    question_id = row["id"] #extract the unique identification code for the question
    question = row["prompt"] #extract the actual Austrian tax law question

    #print current progress to the console (e.g., [1/643])
    print(f"[{i+1}/{len(df)}] Processing: {question_id}")

    answer = get_answer(question) #use the 'get_answer' function to generate a response from your fine-tuned model
    answers.append(answer)
    ids.append(question_id)

    print(f"Preview: {answer[:60]}...") #show the first 60 characters of the answer to monitor the quality in real-time

    time.sleep(0.5)

    #every 50 questions, save a temporary CSV file to your Google Drive to prevent data loss
    if (i + 1) % 50 == 0:
        temp_df = pd.DataFrame({"id": ids, "answer": answers})
        temp_df.to_csv("/content/drive/MyDrive/data/finetuned_results_temp.csv", index=False)
        print(f"\nProgress saved: {i+1} questions done\n")

#save final output
final_df = pd.DataFrame({
    "id": ids,
    "answer": answers
})

final_df.to_csv("/content/drive/MyDrive/data/finetuned_results.csv", index=False)

print("DONE!")
print(f"Total answers: {len(final_df)}")
print(final_df.head(3))

[1/643] Processing: CORP-TAX-001
Preview: Die Körperschaftsteuer basiert auf der Grundlage des § 23 Kö...
[2/643] Processing: CORP-TAX-002
Preview: Wenn eine Körperschaft ein zinsloses Darlehen an einen ihrer...
[3/643] Processing: CORP-TAX-003
Preview: Die §§ 20 bis 23 des EStG binden alle Körperschaften, die al...
[4/643] Processing: CORP-TAX-004
Preview: (a) Bei der Tochtergesellschaft:

1. Die Dividende ist nicht...
[5/643] Processing: CORP-TAX-005
Preview: Die steuerliche Behandlung einer offenen von einer verdeckte...
[6/643] Processing: CORP-TAX-006
Preview: Der maximal mögliche Verlustabzug beträgt 25 Prozent des Ges...
[7/643] Processing: CORP-TAX-007
Preview: Ein Mantelkauf bezieht sich auf den Kauf eines Mantels oder ...
[8/643] Processing: CORP-TAX-008
Preview: Bei einem Forderungsverzicht eines Gesellschafterssteuersatz...
[9/643] Processing: CORP-TAX-009
Preview: Die Verlustverrechnung zwischen ausländischen und inländisch...
[10/643] Processing: CORP-TAX-010
Preview: Ja,